# Project 3

In [ ]:
# Stop any old Spark session
try:
    spark.stop()
except:
    pass

In [ ]:
import os
from pyspark.sql import SparkSession, Window
import pyspark.sql.functions as F

spark = (
    SparkSession.builder
    .appName("Project3-CDC")
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
    .config("spark.sql.catalog.lakehouse", "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.lakehouse.type", "rest")
    .config("spark.sql.catalog.lakehouse.uri", "http://iceberg-rest:8181")
    .config("spark.sql.catalog.lakehouse.warehouse", "s3://warehouse/")
    .config("spark.sql.catalog.lakehouse.io-impl", "org.apache.iceberg.aws.s3.S3FileIO")
    .config("spark.sql.catalog.lakehouse.s3.endpoint", "http://minio:9000")
    .config("spark.sql.catalog.lakehouse.s3.path-style-access", "true")
    .config("spark.sql.catalog.lakehouse.s3.access-key-id", os.environ.get("AWS_ACCESS_KEY_ID", "admin"))
    .config("spark.sql.catalog.lakehouse.s3.secret-access-key", os.environ.get("AWS_SECRET_ACCESS_KEY", "password"))
    .config("spark.sql.catalog.lakehouse.s3.region", "us-east-1")
    .config("spark.sql.defaultCatalog", "lakehouse")
    .config("spark.sql.adaptive.enabled", "false")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print("Spark version:", spark.version)
print("Extensions:", spark.conf.get("spark.sql.extensions"))
print("AQE enabled:", spark.conf.get("spark.sql.adaptive.enabled"))

In [ ]:
# Bronze CDC table
spark.sql("CREATE NAMESPACE IF NOT EXISTS lakehouse.cdc")

spark.sql('''
CREATE TABLE IF NOT EXISTS lakehouse.cdc.bronze_cdc (
    source_table STRING,
    record_id INT,
    op STRING,
    before_json STRING,
    after_json STRING,
    source_lsn BIGINT,
    source_snapshot STRING,
    ts_ms BIGINT,
    kafka_key STRING,
    raw_value STRING,
    topic STRING,
    kafka_partition INT,
    kafka_offset BIGINT,
    kafka_timestamp TIMESTAMP,
    is_tombstone BOOLEAN,
    bronze_ingested_at TIMESTAMP
) USING iceberg
PARTITIONED BY (source_table)
''')

raw_cdc = (
    spark.read
    .format("kafka")
    .option("kafka.bootstrap.servers", "kafka:9092")
    .option("subscribePattern", "dbserver1\\.public\\.(customers|drivers)")
    .option("startingOffsets", "earliest")
    .load()
)

bronze_cdc_batch = (
    raw_cdc
    .select(
        F.regexp_extract(F.col("topic"), r"dbserver1\.public\.(.+)", 1).alias("source_table"),
        F.get_json_object(F.col("key").cast("string"), "$.payload.id").cast("int").alias("key_id"),
        F.get_json_object(F.col("value").cast("string"), "$.payload.after.id").cast("int").alias("after_id"),
        F.get_json_object(F.col("value").cast("string"), "$.payload.before.id").cast("int").alias("before_id"),
        F.get_json_object(F.col("value").cast("string"), "$.payload.op").alias("op"),
        F.get_json_object(F.col("value").cast("string"), "$.payload.before").alias("before_json"),
        F.get_json_object(F.col("value").cast("string"), "$.payload.after").alias("after_json"),
        F.get_json_object(F.col("value").cast("string"), "$.payload.source.lsn").cast("bigint").alias("source_lsn"),
        F.get_json_object(F.col("value").cast("string"), "$.payload.source.snapshot").alias("source_snapshot"),
        F.get_json_object(F.col("value").cast("string"), "$.payload.ts_ms").cast("bigint").alias("ts_ms"),
        F.col("key").cast("string").alias("kafka_key"),
        F.col("value").cast("string").alias("raw_value"),
        F.col("topic"),
        F.col("partition").alias("kafka_partition"),
        F.col("offset").alias("kafka_offset"),
        F.col("timestamp").alias("kafka_timestamp"),
        F.col("value").isNull().alias("is_tombstone"),
        F.current_timestamp().alias("bronze_ingested_at")
    )
    .withColumn("record_id", F.coalesce("after_id", "before_id", "key_id"))
    .drop("key_id", "after_id", "before_id")
)

existing_offsets = (
    spark.table("lakehouse.cdc.bronze_cdc")
    .select("topic", "kafka_partition", "kafka_offset")
)

bronze_cdc_new = (
    bronze_cdc_batch.alias("n")
    .join(
        existing_offsets.alias("e"),
        on=["topic", "kafka_partition", "kafka_offset"],
        how="left_anti"
    )
)

new_rows = bronze_cdc_new.count()

if new_rows > 0:
    bronze_cdc_new.writeTo("lakehouse.cdc.bronze_cdc").append()

print("New Bronze rows appended:", new_rows)

spark.sql('''
SELECT source_table, op, is_tombstone, COUNT(*) AS rows
FROM lakehouse.cdc.bronze_cdc
GROUP BY source_table, op, is_tombstone
ORDER BY source_table, op, is_tombstone
''').show(truncate=False)

In [ ]:
# Inspect Bronze CDC
spark.sql('''
SELECT
  source_table,
  record_id,
  op,
  is_tombstone,
  source_lsn,
  ts_ms,
  kafka_offset
FROM lakehouse.cdc.bronze_cdc
ORDER BY kafka_timestamp DESC, kafka_offset DESC
LIMIT 30
''').show(truncate=False)

In [ ]:
# Create both Silver tables cleanly (if old table exists)
spark.sql("DROP TABLE IF EXISTS lakehouse.cdc.silver_customers")
spark.sql("DROP TABLE IF EXISTS lakehouse.cdc.silver_drivers")

spark.sql('''
CREATE TABLE lakehouse.cdc.silver_customers (
    id INT,
    name STRING,
    email STRING,
    country STRING,
    created_at TIMESTAMP,
    last_updated_ms BIGINT
) USING iceberg
''')

spark.sql('''
CREATE TABLE lakehouse.cdc.silver_drivers (
    id INT,
    name STRING,
    license_number STRING,
    rating DOUBLE,
    city STRING,
    active BOOLEAN,
    created_at TIMESTAMP,
    last_updated_ms BIGINT
) USING iceberg
''')

spark.sql("DESCRIBE lakehouse.cdc.silver_customers").show(truncate=False)
spark.sql("DESCRIBE lakehouse.cdc.silver_drivers").show(truncate=False)

In [ ]:
# Helper for parsing (microseconds issue)
def parse_debezium_ts(json_col, field_name):
    raw = f"get_json_object({json_col}, '$.{field_name}')"
    return F.expr(f"""
        CASE
            WHEN {raw} IS NULL THEN NULL
            WHEN {raw} RLIKE '^[0-9]+$' AND length({raw}) >= 16
                THEN timestamp_micros(CAST({raw} AS BIGINT))
            WHEN {raw} RLIKE '^[0-9]+$' AND length({raw}) >= 13
                THEN timestamp_millis(CAST({raw} AS BIGINT))
            WHEN {raw} RLIKE '^[0-9]+$'
                THEN to_timestamp(from_unixtime(CAST({raw} AS BIGINT)))
            ELSE try_to_timestamp({raw})
        END
    """)

In [ ]:
# Customers: latest event per id, then materialise the merge source
customers_latest = (
    spark.table("lakehouse.cdc.bronze_cdc")
    .filter(F.col("source_table") == "customers")
    .filter(~F.col("is_tombstone"))
    .filter(F.col("record_id").isNotNull())
    .filter(F.col("op").isNotNull())
    .withColumn(
        "rn",
        F.row_number().over(
            Window.partitionBy("record_id").orderBy(
                F.col("ts_ms").desc(),
                F.col("kafka_offset").desc()
            )
        )
    )
    .filter("rn = 1")
    .select(
        F.col("record_id").alias("id"),
        "op",
        F.get_json_object("after_json", "$.name").alias("name"),
        F.get_json_object("after_json", "$.email").alias("email"),
        F.get_json_object("after_json", "$.country").alias("country"),
        parse_debezium_ts("after_json", "created_at").alias("created_at"),
        F.col("ts_ms").alias("last_updated_ms")
    )
)

customers_latest.show(20, truncate=False)
customers_latest.printSchema()

customer_rows = customers_latest.collect()
customers_stage = spark.createDataFrame(customer_rows, customers_latest.schema)
customers_stage.createOrReplaceTempView("customers_cdc_latest_stage")

In [ ]:
# Customers MERGE
spark.sql('''
MERGE INTO lakehouse.cdc.silver_customers AS t
USING customers_cdc_latest_stage AS s
ON t.id = s.id

WHEN MATCHED AND s.op = 'd' THEN DELETE

WHEN MATCHED AND s.op IN ('c', 'u', 'r') THEN UPDATE SET
  t.name = s.name,
  t.email = s.email,
  t.country = s.country,
  t.created_at = s.created_at,
  t.last_updated_ms = s.last_updated_ms

WHEN NOT MATCHED AND s.op IN ('c', 'u', 'r') THEN INSERT (
  id, name, email, country, created_at, last_updated_ms
) VALUES (
  s.id, s.name, s.email, s.country, s.created_at, s.last_updated_ms
)
''')

spark.sql("SELECT COUNT(*) AS silver_customer_rows FROM lakehouse.cdc.silver_customers").show()

spark.sql('''
SELECT *
FROM lakehouse.cdc.silver_customers
ORDER BY id
LIMIT 20
''').show(truncate=False)

In [ ]:
# Drivers: latest event per id, then materialise the merge source
drivers_latest = (
    spark.table("lakehouse.cdc.bronze_cdc")
    .filter(F.col("source_table") == "drivers")
    .filter(~F.col("is_tombstone"))
    .filter(F.col("record_id").isNotNull())
    .filter(F.col("op").isNotNull())
    .withColumn(
        "rn",
        F.row_number().over(
            Window.partitionBy("record_id").orderBy(
                F.col("ts_ms").desc(),
                F.col("kafka_offset").desc()
            )
        )
    )
    .filter("rn = 1")
    .select(
        F.col("record_id").alias("id"),
        "op",
        F.get_json_object("after_json", "$.name").alias("name"),
        F.get_json_object("after_json", "$.license_number").alias("license_number"),
        F.get_json_object("after_json", "$.rating").cast("double").alias("rating"),
        F.get_json_object("after_json", "$.city").alias("city"),
        F.get_json_object("after_json", "$.active").cast("boolean").alias("active"),
        F.to_timestamp(F.get_json_object("after_json", "$.created_at")).alias("created_at"),
        F.col("ts_ms").alias("last_updated_ms")
    )
)

driver_rows = drivers_latest.collect()
drivers_stage = spark.createDataFrame(driver_rows, drivers_latest.schema)
drivers_stage.createOrReplaceTempView("drivers_cdc_latest_stage")

drivers_stage.show(20, truncate=False)
drivers_stage.printSchema()

In [ ]:
# Drivers MERGE
spark.sql('''
MERGE INTO lakehouse.cdc.silver_drivers AS t
USING drivers_cdc_latest_stage AS s
ON t.id = s.id

WHEN MATCHED AND s.op = 'd' THEN DELETE

WHEN MATCHED AND s.op IN ('c', 'u', 'r') THEN UPDATE SET
  t.name = s.name,
  t.license_number = s.license_number,
  t.rating = s.rating,
  t.city = s.city,
  t.active = s.active,
  t.created_at = s.created_at,
  t.last_updated_ms = s.last_updated_ms

WHEN NOT MATCHED AND s.op IN ('c', 'u', 'r') THEN INSERT (
  id, name, license_number, rating, city, active, created_at, last_updated_ms
) VALUES (
  s.id, s.name, s.license_number, s.rating, s.city, s.active, s.created_at, s.last_updated_ms
)
''')

spark.sql("SELECT COUNT(*) AS silver_driver_rows FROM lakehouse.cdc.silver_drivers").show()

spark.sql('''
SELECT *
FROM lakehouse.cdc.silver_drivers
ORDER BY id
LIMIT 20
''').show(truncate=False)

In [ ]:
# Validate Silver row counts against PostgreSQL
import psycopg2

conn = psycopg2.connect(
    host=os.environ.get("PG_HOST", "postgres"),
    port=int(os.environ.get("PG_PORT", 5432)),
    dbname=os.environ.get("PG_DB", "sourcedb"),
    user=os.environ.get("PG_USER", "cdc_user"),
    password=os.environ.get("PG_PASSWORD", "cdc_pass"),
)
conn.autocommit = True
cur = conn.cursor()

for source_table, silver_table in [
    ("customers", "lakehouse.cdc.silver_customers"),
    ("drivers", "lakehouse.cdc.silver_drivers"),
]:
    cur.execute(f"SELECT COUNT(*) FROM {source_table}")
    pg_count = cur.fetchone()[0]
    silver_count = spark.table(silver_table).count()
    print(f"{source_table}: postgres={pg_count}, silver={silver_count}")

cur.close()
conn.close()

In [ ]:
# Snapshot history for the report
spark.sql('''
SELECT *
FROM lakehouse.cdc.silver_customers.history
ORDER BY made_current_at DESC
''').show(truncate=False)

spark.sql('''
SELECT *
FROM lakehouse.cdc.silver_drivers.history
ORDER BY made_current_at DESC
''').show(truncate=False)